# MIMO基础

**MIMO Basics**

---

本notebook介绍多输入多输出（Multiple-Input Multiple-Output, MIMO）无线通信技术的基本概念。MIMO是现代无线通信系统的核心技术之一，在WiFi（802.11n/ac/ax）、4G LTE、5G NR等技术中发挥关键作用。理解MIMO的空间复用和波束成形原理对于学习OTFS在MIMO系统中的应用至关重要。

## 1. 目标 (Objectives)

通过本notebook，您将：

- **理解空间复用原理**：掌握MIMO如何利用多天线同时传输多个数据流
- **掌握MIMO信道容量**：理解信道容量与天线配置和SNR的关系
- **认识多用户MIMO（MU-MIMO）**：了解如何通过MIMO服务多个用户
- **了解大规模MIMO（Massive MIMO）**：认识天线数量大幅增加后的特性和优势
- **理解预编码与均衡基础**：掌握MIMO系统中的信号处理技术
- **为学习OTFS打下基础**：OTFS在延迟-多普勒域处理MIMO信道具有独特优势

## 2. MIMO直观理解 (Conceptual Understanding)

### 2.1 从单天线到多天线

传统单天线（SISO）系统：
- 发射机1根天线，接收机1根天线
- 同一时刻只能传输一个数据流
- 信道容量受限于香农极限

MIMO系统利用多天线技术：
- 发射机多根天线，接收机多根天线
- 可以同时传输多个独立数据流
- 大幅提升系统容量和可靠性

### 2.2 空间复用原理

空间复用（Spatial Multiplexing）是MIMO的核心技术之一。其基本思想是：

- **相同频率、相同时间**：利用不同天线对之间的独立信道
- **并行传输**：将数据分成多个流，同时通过不同天线发送
- **容量提升**：理论容量随最小天线数线性增长

以2x2 MIMO为例：

```
发射机                信道H                  接收机
  TX1 ──────────────────────────────── RX1
    \                     H11                    /
     \         H12  ─────────────  H22         /
      \              ↕↔↕↔↕                   /
       \            H21                      /
        TX2 ─────────────────────────────── RX2
```

发射信号：
$$\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$$

接收信号：
$$\mathbf{y} = \mathbf{H} \mathbf{x} + \mathbf{n}$$

其中 $\mathbf{H}$ 是2x2信道矩阵：
$$\mathbf{H} = \begin{bmatrix} H_{11} & H_{12} \\ H_{21} & H_{22} \end{bmatrix}$$

### 2.3 多用户MIMO（MU-MIMO）

MU-MIMO（Multi-User MIMO）允许发射机同时服务多个用户：

| 特性 | SU-MIMO（单用户） | MU-MIMO（多用户） |
|------|------------------|-------------------|
| 同时传输 | 同一用户的多个流 | 不同用户的独立流 |
| 天线配置 | 发射机多天线对应单用户 | 基站多天线对应多用户 |
| 复杂度 | 接收机复杂 | 基站预编码复杂 |
| 调度 | 简单 | 需要用户调度 |

**MU-MIMO的关键技术**：

1. **预编码（Precoding）**：基站利用信道信息对信号进行预处理，消除用户间干扰
2. **用户调度**：基站根据信道条件选择最佳用户组合
3. **信道状态信息（CSI）**：需要知道各用户到基站的海量CSI

### 2.4 大规模MIMO（Massive MIMO）

Massive MIMO是5G的关键技术之一，采用大规模天线阵列：

- **天线数量**：32/64/128甚至更多（相比传统MIMO的2/4/8根）
- **波束成形**：极窄波束精准指向用户
- **空间分辨率**：大幅提升，可同时服务更多用户

**Massive MIMO的优势**：

| 优势 | 说明 |
|------|------|
| **容量提升** | 大量天线支持更多并行流 |
| **频谱效率** | 窄波束减少干扰，提高频率复用 |
| **能量效率** | 定向波束将能量集中在用户方向 |
| **覆盖增强** | 波束增益扩大小区覆盖范围 |

**大规模天线阵列的极限**：

当天线数趋向无穷时：
1. 热噪声被自然平均
2. 简单线性处理（如最大比合并）可达最优
3. 信道变得渐进确定性

## 3. 代码演示：MIMO信道容量 (Channel Capacity)

下面通过代码演示MIMO信道容量的计算和可视化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set up Chinese font support
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

### 3.1 MIMO信道容量基础

对于给定信道矩阵 $\mathbf{H}$，MIMO信道容量（bits/s/Hz）为：

$$C = \log_2 \det\left(\mathbf{I}_N + \frac{\rho}{M} \mathbf{H} \mathbf{H}^H\right)$$

其中：
- $\rho$：接收端SNR
- $M$：发射天线数
- $N$：接收天线数
- $\mathbf{H}^H$：$\mathbf{H}$的厄米特转置

In [ ]:
def mimo_capacity(H, SNR_db):
    """
    Calculate MIMO channel capacity
    
    Parameters:
    -----------
    H : ndarray
        Channel matrix of shape (N_rx, N_tx)
    SNR_db : float
        SNR in dB
    
    Returns:
    -------
    float : Capacity in bits/s/Hz
    """
    N_rx, N_tx = H.shape
    SNR = 10 ** (SNR_db / 10)
    
    # Capacity formula: C = log2(det(I + SNR/M * H*H^H))
    H_H = H.conj().T  # Hermitian transpose
    matrix = np.eye(N_rx) + (SNR / N_tx) * H @ H_H
    
    # Eigenvalue decomposition for numerical stability
    eigenvalues = np.linalg.eigvalsh(matrix)
    eigenvalues = np.maximum(eigenvalues, 1e-10)  # Avoid log of non-positive
    
    capacity = np.sum(np.log2(eigenvalues))
    return capacity


# Example: 2x2 MIMO channel
np.random.seed(42)
H_2x2 = np.random.randn(2, 2) + 1j * np.random.randn(2, 2)

# Normalize channel matrix
H_2x2 = H_2x2 / np.sqrt(np.mean(np.abs(H_2x2)**2))

print(f"2x2 MIMO Channel Matrix H:")
print(H_2x2)
print(f"\nChannel singular values: {np.linalg.svd(H_2x2, compute_uv=False)}")
print(f"Channel condition number: {np.linalg.cond(H_2x2):.2f}")

### 3.2 MIMO容量 vs SNR曲线

In [ ]:
# Compare MIMO capacity for different antenna configurations
SNR_db = np.linspace(-10, 30, 100)

# Antenna configurations: (N_tx, N_rx)
configs = [
    (1, 1),   # SISO as baseline
    (2, 2),   # 2x2 MIMO
    (4, 4),   # 4x4 MIMO
    (8, 8),   # 8x8 Massive MIMO
]

# Calculate capacity for each configuration (average over many channel realizations)
n_iterations = 500

capacity_avg = {}
capacity_std = {}

for N_tx, N_rx in configs:
    capacities = np.zeros((n_iterations, len(SNR_db)))
    for i in range(n_iterations):
        # Generate random i.i.d. Rayleigh fading channel
        H = np.random.randn(N_rx, N_tx) + 1j * np.random.randn(N_rx, N_tx)
        H = H / np.sqrt(2)  # Normalize
        
        for j, snr in enumerate(SNR_db):
            capacities[i, j] = mimo_capacity(H, snr)
    
    capacity_avg[(N_tx, N_rx)] = np.mean(capacities, axis=0)
    capacity_std[(N_tx, N_rx)] = np.std(capacities, axis=0)

print("Capacity calculations completed for all configurations.")

In [ ]:
# Plot MIMO capacity vs SNR
fig, ax = plt.subplots(figsize=(12, 8))

colors = ['gray', 'blue', 'green', 'red']
markers = ['o', 's', '^', 'D']

for idx, (N_tx, N_rx) in enumerate(configs):
    label = f'{N_tx}x{N_rx} MIMO'
    if N_tx == 1 and N_rx == 1:
        label = 'SISO (baseline)'
    elif N_tx >= 8:
        label = f'{N_tx}x{N_rx} Massive MIMO'
    
    ax.plot(SNR_db, capacity_avg[(N_tx, N_rx)], 
            color=colors[idx], linewidth=2, label=label)
    ax.fill_between(SNR_db, 
                    capacity_avg[(N_tx, N_rx)] - capacity_std[(N_tx, N_rx)],
                    capacity_avg[(N_tx, N_rx)] + capacity_std[(N_tx, N_rx)],
                    color=colors[idx], alpha=0.2)

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('Channel Capacity (bits/s/Hz)', fontsize=12)
ax.set_title('MIMO Channel Capacity vs SNR for Different Antenna Configurations', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-10, 30)
ax.set_ylim(0, 35)

plt.tight_layout()
plt.show()

print("\n观察：")
print("  - MIMO容量随SNR增加而增加，遵循对数关系")
print("  - 更多天线提供更高的容量（在同一SNR下）")
print("  - 高SNR区域，容量增长趋于线性（与min(N_tx,N_rx)成正比）")
print("  - 阴影区域表示不同信道实现的容量变化范围")

### 3.3 空间复用增益与分集增益

MIMO提供两种主要增益：

| 增益类型 | 说明 | 效果 |
|----------|------|------|
| **空间复用** | 多个独立数据流并行传输 | 容量线性增长 |
| **分集增益** | 同一数据通过多路径传输 | 可靠性提升，抗衰落 |

空间复用利用信道的平行子信道：
$$\mathbf{H} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^H$$

其中 $\mathbf{\Sigma}$ 包含奇异值 $\sigma_1, \sigma_2, ...$

In [ ]:
# Visualize singular values of MIMO channel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Singular values distribution for different antenna configs
N_tx = 4
N_rx = 4
n_channels = 1000

all_singular_values = []
for _ in range(n_channels):
    H = np.random.randn(N_rx, N_tx) + 1j * np.random.randn(N_rx, N_tx)
    H = H / np.sqrt(2)
    singular_values = np.linalg.svd(H, compute_uv=False)
    all_singular_values.append(singular_values)

all_singular_values = np.array(all_singular_values)

box_data = [all_singular_values[:, i] for i in range(N_tx)]
bp = axes[0].boxplot(box_data, labels=[f'sv_{i+1}' for i in range(N_tx)], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

axes[0].set_xlabel('Singular Value Index', fontsize=11)
axes[0].set_ylabel('Singular Value Magnitude', fontsize=11)
axes[0].set_title(f'Distribution of Singular Values ({N_tx}x{N_rx} MIMO)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Right plot: Capacity comparison - Spatial复用 vs SISO
SNR_db = np.linspace(-5, 25, 50)

# Theoretical SISO capacity
siso_capacity = np.log2(1 + 10 ** (SNR_db / 10))

# 2x2 MIMO capacity (known result: 2 * log2(1 + SNR) at high SNR approximately)
np.random.seed(0)
mimo_capacities = []
for snr in SNR_db:
    H = np.random.randn(2, 2) + 1j * np.random.randn(2, 2)
    H = H / np.sqrt(2)
    mimo_capacities.append(mimo_capacity(H, snr))

axes[1].plot(SNR_db, siso_capacity, 'k--', linewidth=2, label='SISO')
axes[1].plot(SNR_db, mimo_capacities, 'b-', linewidth=2, label='2x2 MIMO')
axes[1].plot(SNR_db, 2 * siso_capacity, 'r:', linewidth=2, label='2x SISO (theoretical max)')

axes[1].set_xlabel('SNR (dB)', fontsize=11)
axes[1].set_ylabel('Capacity (bits/s/Hz)', fontsize=11)
axes[1].set_title('Spatial Multiplexing Gain of 2x2 MIMO', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n关键观察：")
print("  - 奇异值分布显示不同空间信道的强度差异")
print("  - 2x2 MIMO在低SNR下接近SISO容量（复用增益受限）")
print("  - 2x2 MIMO在高SNR下可达2倍SISO容量（完全空间复用）")

## 4. 预编码与均衡基础 (Precoding and Equalization)

### 4.1 为什么要预编码？

在MIMO系统中，多个数据流同时传输会相互干扰：

- **同用户多流**：同一用户的多个流之间可能产生干扰
- **多用户干扰**：MU-MIMO中不同用户的流相互干扰

预编码技术在发射端对信号进行预处理，利用信道状态信息（CSI）将干扰转化为有益信号或消除干扰。

### 4.2 线性预编码方案

| 方案 | 原理 | 复杂度 | 性能 |
|------|------|--------|------|
| **迫零（ZF）** | 完全消除干扰 | 中等 | 噪声放大 |
| **最小均方误差（MMSE）** | 平衡干扰消除与噪声 | 中等 | 最优非线性 |
| **最大比传输（MRT）** | 最大化目标信号 | 低 | 仅适用单用户 |
| **脏纸编码（DPC）** | 已知干扰时最优 | 高 | 香农容量可达 |

**迫零预编码**：
$$\mathbf{F}_{ZF} = \mathbf{H}^{-1} \text{（当 } N_t = N_r \text{）}$$

**MMSE预编码**：
$$\mathbf{F}_{MMSE} = \mathbf{H}^H (\mathbf{H}\mathbf{H}^H + \sigma^2 \mathbf{I})^{-1}$$

In [ ]:
# Demonstrate precoding: ZF and MMSE
def zf_precoder(H):
    """Zero-Forcing precoder"""
    return np.linalg.pinv(H)  # Pseudoinverse

def mmse_precoder(H, SNR_db):
    """MMSE precoder"""
    N_tx, N_rx = H.shape[1], H.shape[0]
    SNR = 10 ** (SNR_db / 10)
    sigma2 = 1 / SNR
    return H.conj().T @ np.linalg.inv(H @ H.conj().T + sigma2 * np.eye(N_rx))


# Example: 2x2 MIMO with interference
np.random.seed(123)
H = np.random.randn(2, 2) + 1j * np.random.randn(2, 2)
H = H / np.sqrt(2)

# Transmit symbols
s = np.array([1 + 1j, -1 - 1j])  # Two independent streams

# ZF Precoding
F_zf = zf_precoder(H)
x_zf = F_zf @ s
y_zf = H @ x_zf

# MMSE Precoding
F_mmse = mmse_precoder(H, 10)  # 10dB SNR
x_mmse = F_mmse @ s
y_mmse = H @ x_mmse

print("Transmitted symbols:", s)
print("\nZF Precoding:")
print("  Precoded signals:", x_zf)
print("  Received signals:", y_zf)
print("  Error (should be ~0):", np.linalg.norm(y_zf - s))

print("\nMMSE Precoding (SNR=10dB):")
print("  Precoded signals:", x_mmse)
print("  Received signals:", y_mmse)
print("  Error (not zero due to noise consideration):", np.linalg.norm(y_mmse - s))

### 4.3 MIMO均衡

接收机通过均衡器恢复发送信号。常见的均衡策略：

**匹配滤波器（MF）**：
$$\hat{s} = \mathbf{H}^H \mathbf{y}$$

**迫零均衡器**：
$$\hat{s} = \mathbf{H}^{-1} \mathbf{y}$$

**MMSE均衡器**：
$$\hat{s} = (\mathbf{H}^H \mathbf{H} + \sigma^2 \mathbf{I})^{-1} \mathbf{H}^H \mathbf{y}$$

In [ ]:
# Demonstrate MIMO equalization
def mf_equalizer(H, y):
    """Matched Filter equalizer"""
    return H.conj().T @ y

def zf_equalizer(H, y):
    """Zero-Forcing equalizer"""
    return np.linalg.solve(H, y)

def mmse_equalizer(H, y, SNR_db):
    """MMSE equalizer"""
    N_tx = H.shape[1]
    SNR = 10 ** (SNR_db / 10)
    sigma2 = 1 / SNR
    return np.linalg.solve(H.conj().T @ H + sigma2 * np.eye(N_tx), H.conj().T @ y)


# Simulation: BER vs SNR for different equalizers
np.random.seed(42)
SNR_db_range = np.arange(0, 31, 5)
n_symbols = 10000

ber_mf = []
ber_zf = []
ber_mmse = []

for SNR_db in SNR_db_range:
    errors_mf = 0
    errors_zf = 0
    errors_mmse = 0
    
    for _ in range(100):
        # Random channel
        H = (np.random.randn(2, 2) + 1j * np.random.randn(2, 2)) / np.sqrt(2)
        
        # Random QPSK symbols
        s = (np.random.choice([1+1j, 1-1j, -1+1j, -1-1j], size=2)) / np.sqrt(2)
        
        # Channel + noise
        SNR = 10 ** (SNR_db / 10)
        noise_var = 1 / SNR
        n = np.sqrt(noise_var/2) * (np.random.randn(2) + 1j * np.random.randn(2))
        y = H @ s + n
        
        # Equalize
        s_hat_mf = mf_equalizer(H, y)
        s_hat_zf = zf_equalizer(H, y)
        s_hat_mmse = mmse_equalizer(H, y, SNR_db)
        
        # Count errors (QPSK decision)
        def qpsk_decision(x):
            return np.sign(np.real(x)) + 1j * np.sign(np.imag(x))
        
        errors_mf += np.sum(qpsk_decision(s_hat_mf) != qpsk_decision(s))
        errors_zf += np.sum(qpsk_decision(s_hat_zf) != qpsk_decision(s))
        errors_mmse += np.sum(qpsk_decision(s_hat_mmse) != qpsk_decision(s))
    
    ber_mf.append(errors_mf / (2 * 100))
    ber_zf.append(errors_zf / (2 * 100))
    ber_mmse.append(errors_mmse / (2 * 100))

print("Equalization comparison completed.")

In [ ]:
# Plot BER comparison
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(SNR_db_range, ber_mf, 'b-o', linewidth=2, markersize=8, label='Matched Filter')
ax.semilogy(SNR_db_range, ber_zf, 'g-s', linewidth=2, markersize=8, label='Zero-Forcing')
ax.semilogy(SNR_db_range, ber_mmse, 'r-^', linewidth=2, markersize=8, label='MMSE')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('Bit Error Rate (BER)', fontsize=12)
ax.set_title('MIMO Equalizer Performance Comparison (2x2 QPSK)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0, 30)
ax.set_ylim(1e-3, 1)

plt.tight_layout()
plt.show()

print("\n观察：")
print("  - MMSE均衡器在高噪声区域表现最好")
print("  - ZF均衡器在高SNR区域接近MMSE性能")
print("  - MF性能最差，因为它没有考虑信道矩阵的逆")

## 5. 关联OTFS：OTFS的MU-MIMO优势

### 5.1 OTFS如何处理MIMO信道

OTFS（正交时频空）在**延迟-多普勒域**处理MIMO信道，与传统时频域MIMO相比具有独特优势：

| 特性 | 时频域MIMO | OTFS MIMO |
|------|-----------|-----------|
| **信道表示** | 时变频率响应 $\mathbf{H}(f,t)$ | 时不变等价信道 $\mathbf{H}(\tau, \nu)$ |
| **处理方式** | 每个子载波独立处理 | 整个带宽联合处理 |
| **信道条件数** | 随频率变化，可能很差 | 条件数更稳定 |
| **均衡复杂度** | 需处理频率选择性 | 接近单抽头均衡 |
| **多普勒处理** | 高多普勒导致子载波干扰 | 天然兼容高多普勒 |

### 5.2 OTFS的MIMO优势

**1. 更好的信道条件数**

在时频域，不同子载波经历不同的信道增益，导致某些子载流的SNR极低。OTFS通过延迟-多普勒到时频的变换，使得：

- 所有时频资源经历相同的等价信道
- 信道条件数显著改善
- 均衡器设计大大简化

**2. 简化的预编码**

由于OTFS等价信道接近时不变：

- 发射机可以获得更准确的CSI
- 预编码矩阵设计更简单
- MU-MIMO调度更有效

**3. 高移动性支持**

OTFS在延迟-多普勒域处理多普勒：

- 每条多径的多普勒频移独立表征
- 高速移动不破坏子载波正交性
- 适合车联网（V2X）、高铁等场景

### 5.3 OTFS MIMO容量优势

理论分析表明，在高移动性场景下：

$$\text{OTFS容量} \geq \text{OFDM容量} \cdot \text{多普勒分集增益}$$

关键点：

- **多普勒分集**：OTFS将多普勒扩展转化为可利用的分集增益
- **频率复用**：更好的条件数允许更密集的频率复用
- **MU-MIMO效率**：信道稳定性提高了多用户调度的准确性

In [ ]:
# Illustrate OTFS vs OFDM channel condition number
# Simulate time-varying channel and compare condition numbers

np.random.seed(100)

# OFDM: Channel changes across subcarriers (frequency selectivity)
# Each subcarrier sees a different channel realization

n_subcarriers = 64
n_paths = 5
SNR_db = 20

# Generate frequency-selective channel for OFDM
H_ofdm_subcarriers = []
for sc in range(n_subcarriers):
    # Each subcarrier has slightly different channel
    H_sc = np.zeros((2, 2))
    for p in range(n_paths):
        # Random path with phase depending on subcarrier index
        phase = 2 * np.pi * sc * p / n_subcarriers
        alpha = np.random.randn() + 1j * np.random.randn()
        H_sc += alpha * np.exp(-1j * phase) * np.eye(2)
    H_ofdm_subcarriers.append(H_sc)

H_ofdm_subcarriers = np.array(H_ofdm_subcarriers)

# Condition numbers across subcarriers
cond_ofdm = np.array([np.linalg.cond(H) for H in H_ofdm_subcarriers])

# OTFS: Equivalent channel is the average/total channel (more stable)
# OTFS transforms to DD domain,信道条件数更稳定
H_otfs = np.mean(H_ofdm_subcarriers, axis=0)  # Simplified model
cond_otfs = np.linalg.cond(H_otfs)

print(f"OFDM subcarrier condition numbers: min={cond_ofdm.min():.2f}, max={cond_ofdm.max():.2f}, mean={cond_ofdm.mean():.2f}")
print(f"OTFS equivalent channel condition number: {cond_otfs:.2f}")
print(f"\nOTFS condition number improvement: {cond_ofdm.mean() / cond_otfs:.2f}x")

In [ ]:
# Visualize condition number comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: OFDM condition number across subcarriers
axes[0].plot(range(n_subcarriers), cond_ofdm, 'b-', linewidth=1.5)
axes[0].axhline(y=cond_otfs, color='r', linestyle='--', linewidth=2, label=f'OTFS equiv: {cond_otfs:.1f}')
axes[0].set_xlabel('Subcarrier Index', fontsize=11)
axes[0].set_ylabel('Condition Number', fontsize=11)
axes[0].set_title('OFDM Channel Condition Number Across Subcarriers', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Capacity comparison for same total power
SNR_db_range = np.linspace(-5, 25, 50)
SNR = 10 ** (SNR_db_range / 10)

# OFDM: Use mean condition number for capacity estimate
mean_cond = np.mean(cond_ofdm)
# Approximate capacity reduction due to poor conditioning
capacity_ofdm_approx = 2 * np.log2(1 + SNR / mean_cond)  # Simplified model

# OTFS: Better conditioning, full spatial multiplexing
capacity_otfs = 2 * np.log2(1 + SNR / cond_otfs * mean_cond)  # Modeled improvement

axes[1].plot(SNR_db_range, capacity_ofdm_approx, 'b-', linewidth=2, label='OFDM (approx)')
axes[1].plot(SNR_db_range, capacity_otfs, 'r-', linewidth=2, label='OTFS (approx)')
axes[1].plot(SNR_db_range, 2 * np.log2(1 + SNR), 'k:', linewidth=2, label='Ideal 2x2')

axes[1].set_xlabel('SNR (dB)', fontsize=11)
axes[1].set_ylabel('Capacity (bits/s/Hz)', fontsize=11)
axes[1].set_title('Capacity Comparison: OTFS vs OFDM in Frequency-Selective Channel', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n观察：")
print("  - OFDM在不同子载波上信道条件数变化大")
print("  - OTFS的等效信道条件数更稳定（所有时频资源共享）")
print("  - 更好的信道条件数带来更高的有效容量")

### 5.4 延迟-多普勒域MIMO处理

OTFS在延迟-多普勒域处理MIMO的核心思想：

1. **信道表示**：
   $$h(\tau, \nu) = \sum_{k=0}^{K-1} \alpha_k \delta(\tau - \tau_k) \delta(\nu - \nu_k)$$
   
   每条多径对应一个独立的延迟-多普勒冲激

2. **发送信号**：
   在延迟-多普勒域放置MIMO天线阵的QAM符号
   $$\mathbf{X}(\tau, \nu) = \begin{bmatrix} X_1(\tau, \nu) & X_2(\tau, \nu) \end{bmatrix}^T$$

3. **接收处理**：
   通过二维变换（Zak变换）将延迟-多普勒域映射到时频域
   - 每条多径在时频域表现为独立的贡献
   - 各天线路径可以分离处理
   - 均衡变为简单的匹配滤波

**关键优势**：
- 多径不再叠加成频率选择性衰落，而是保持分离
- 多普勒扩展被显式建模，不导致正交性丧失
- MIMO均衡在延迟域变为稀疏矩阵求逆

## 6. 思考题 (Review Questions)

1. **空间复用原理**：解释为什么MIMO的空间复用增益与信道矩阵的条件数密切相关。如果信道矩阵的两个奇异值相差很大（条件数很大），空间复用增益会受到什么影响？

2. **容量计算**：假设一个4x4 MIMO系统，在20dB SNR下工作。使用随机i.i.d.信道模型，计算理论信道容量（bits/s/Hz）。如果天线数增加到8x8，容量如何变化？

3. **预编码设计**：在MU-MIMO系统中，基站需要同时向两个用户发送数据。用户1的信道为 $\mathbf{H}_1 = [1, 0.5]$，用户2的信道为 $\mathbf{H}_2 = [0.3, 1]$。设计一个简单的ZF预编码方案来消除用户间干扰。

4. **大规模MIMO特性**：当发射天线数量趋向无穷时，为什么简单线性处理（如最大比传输）可以达到次优性能？这与"硬化"现象有什么关系？

5. **OTFS vs MIMO-OFDM**：在高速移动场景下，比较MIMO-OFDM和OTFS在处理高多普勒扩展时的复杂度差异。为什么OTFS在高移动性场景下更受欢迎？

6. **条件数与均衡**：解释为什么OTFS在延迟-多普勒域处理MIMO信道时通常具有更好的条件数。这个特性如何帮助简化均衡器设计？

7. **实际应用**：在5G NR系统中，使用大规模MIMO（64T64R）和波束成形技术。如果要覆盖一个半径为500米的城区小区，请分析：
   - 波束成形如何扩大覆盖范围？
   - 空间复用如何在覆盖区域内提升容量？
   - 多用户调度如何影响系统整体性能？

---

## 总结 (Summary)

本notebook介绍了MIMO无线通信的基础知识：

- **空间复用原理**：利用多天线同时传输多个独立数据流，容量随天线数线性增长
- **MIMO信道容量**：基于香农公式，通过信道矩阵的奇异值分解计算
- **MU-MIMO**：通过预编码技术同时服务多个用户，提升系统整体吞吐量
- **Massive MIMO**：大规模天线阵列实现精准波束成形和极高空间分辨率
- **预编码与均衡**：ZF、MMSE等线性技术平衡干扰消除与噪声放大
- **OTFS关联**：OTFS在延迟-多普勒域处理MIMO，提供更好的信道条件数和更简单的均衡

这些概念为理解OTFS调制技术在MIMO系统中的应用提供了基础。